# Analítica: de bronce a la tabla de tres columnas

Este notebook recorre el hito 8 de EDsim. El hito 7 dejó en Postgres **lo registrado** tal como
llega (la capa **bronce**), con sus despistes: retrasos, dobles clics, valores erróneos, olvidos,
relojes desajustados y correcciones tardías. El hito 8 lo transforma con **dbt** en capas, como
un hospital de verdad:

- **preparación**: una fila por registro (sin las relecturas de Kafka) y, en
  `registros_vigentes`, la **limpieza**: los duplicados de un doble clic se quitan y las
  correcciones se aplican;
- **plata**: ocho tablas con la forma de un HIS (episodio, triaje, valoración, prueba…);
- **oro**: los indicadores, el censo y los avisos de calidad del dato.

Y como EDsim tiene **la verdad** (lo que pasó de verdad), puede hacer lo que un hospital no
puede: medir cuánto se equivocan los indicadores con datos sucios y cuánto recupera la limpieza.

El notebook cuenta, en este orden:

2. **Lucía en memoria**: el escenario L del notebook 07, con sus siete registros y su libreta.
3. **La carga** (solo en modo normal): una ejecución de 14 días con tres perfiles, en bronce
   sin Kafka, la verdad en Postgres y las cuatro construcciones de dbt.
4. **Lucía atraviesa las capas**: bronce, la limpieza, plata y oro.
5. **La tabla de tres columnas**: verdad, ingenuo y limpio.
6. **Lo que no se arregla**: los relojes, los olvidos y los valores erróneos sin corregir.
7. **Cada regla frente a la libreta**: la limpieza quita y corrige exactamente lo que apuntó el
   registro clínico.

Toda la lógica vive en el paquete y en el proyecto dbt, con tests (`edsim.store`,
`edsim.runs`, `analitica/`); aquí solo se llama a sus funciones, se consulta con SQL, se dibuja
y se explica (decisión 18).

## 1. Cómo usar este notebook

**Modo normal** (lo habitual en local; hace falta Docker). Para las secciones 3 a 7:

1. En una terminal, en la raíz del proyecto: `docker compose up --wait postgres` (basta
   Postgres: el notebook no usa Kafka ni Grafana).
2. `uv sync --all-packages --extra postgres`: instala `psycopg` en `.venv`. Ojo: un `uv sync`
   sin el extra lo vuelve a quitar.
3. La primera vez, `cd analitica && uv sync --locked`: el entorno de dbt, que es un proyecto uv
   aparte con su propio `uv.lock` (necesita red).
4. Abre el notebook en Cursor con el kernel de `.venv` (*Select Kernel*) y usa
   *Restart & Run All*.

Las migraciones de bronce y de la verdad las aplica el propio notebook
(`edsim almacen preparar`), antes de escribir nada.

El notebook escribe en la base **`edsim`**, la de compose, con identificadores propios
(`nb08_…`, con la fecha y la hora): no se mezcla con otras ejecuciones ni con otra vez que lo
ejecutes. Al terminar, abre Grafana (`docker compose --profile analitica up --wait`,
http://127.0.0.1:3000) y verás la ejecución del notebook en los tres cuadros. El notebook no
borra sus filas: al final imprime la consulta para hacerlo.

**Modo rápido** (`EDSIM_NB_RAPIDO=1`): sin Docker. La sección 2 corre entera en memoria y las
celdas que necesitan Postgres y dbt se saltan con un aviso. Es lo que hace el CI
(`EDSIM_NB_RAPIDO=1 uv run pytest --nbmake notebooks/`).

- **Duración:** en modo rápido, unos segundos; en modo normal, unos minutos (la celda de la
  carga imprime cuánto tarda cada paso).
- **Las cifras que cita el texto** son las de los valores por defecto (14 días, 2 de
  calentamiento, semilla 2026). Si cambias un parámetro, las cifras del texto ya no valen, pero
  los `assert` siguen comprobando lo que el texto afirma.
- **Las gráficas desaparecen tras cada commit:** el hook `nbstripout` borra los resultados.

In [ ]:
import dataclasses
import json
import os
import shutil
import subprocess
import sys
import tempfile
import time
from collections import Counter
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

import matplotlib.pyplot as plt

from edsim.analytics import dbt_environment
from edsim.clinical_record import Roll, Timed, load_record_profiles, record, record_json
from edsim.engine.events import EventLog, MemorySink
from edsim.engine.report import MUTED_COLOR, OBSERVED_COLOR, THEORY_COLOR
from edsim.live.directo import REPLAY, RecordInfo, RunInfo
from edsim.live.launch import log_start_date
from edsim.runs import RunStore, read_recorded_records, read_recorded_run, record_run, simulate_run
from edsim.store.offline import bronze_messages
from edsim.store.postgres import (
    DEFAULT_URL,
    MissingPostgresExtraError,
    connect,
    import_psycopg,
    write_bronze,
)

INICIO = time.perf_counter()

# --- Parámetros ------------------------------------------------------------------------
# Con EDSIM_NB_RAPIDO=1 (el CI) solo corre lo que va en memoria: sin Docker, Postgres ni dbt.
RAPIDO = os.environ.get("EDSIM_NB_RAPIDO") == "1"

RAIZ = Path("..").resolve()  # el notebook se ejecuta desde notebooks/
CONFIG = RAIZ / "config"
ANALITICA = RAIZ / "analitica"
# La base edsim de compose.yaml: la misma que usa dbt con los valores por defecto de
# analitica/profiles.yml (la celda de la carga le quita las variables EDSIM_PG_* heredadas).
POSTGRES = DEFAULT_URL
SEMILLA = 2026
DIAS, CAL = 14, 2  # catorce días: dos de calentamiento y doce de ventana
HORIZONTE = DIAS * 1440
CALENTAMIENTO = CAL * 1440
SELLO = f"{datetime.now():%Y%m%dT%H%M%S}"
RUN_ID = f"nb08_{SELLO}"  # la ejecución: propia, no se mezcla con otras
LUCIA = f"nb08l_{SELLO}"  # la verdad de Lucía; lo registrado será {LUCIA}_realista_s7
EDSIM = Path(sys.executable).with_name("edsim")  # la CLI del mismo entorno que el kernel
UV = shutil.which("uv")  # dbt se lanza con «uv run», en su propio entorno
PERFILES = ("limpio", "realista", "hostil")
COLOR = {"limpio": MUTED_COLOR, "realista": OBSERVED_COLOR, "hostil": THEORY_COLOR}
CET = timezone(timedelta(hours=1))  # el huso de event_time en EDsim

if not RAPIDO:
    try:
        import_psycopg()
    except MissingPostgresExtraError as error:
        raise RuntimeError(
            f"{error}, y reinicia el kernel; o ejecuta el notebook en modo rápido "
            "(EDSIM_NB_RAPIDO=1)"
        ) from None
    if UV is None:
        raise RuntimeError("No se encuentra uv en el PATH: dbt se lanza con «uv run».")

PERFILES_REGISTRO = load_record_profiles(CONFIG / "registro_clinico.toml")
print(f"Modo {'rápido' if RAPIDO else 'normal'} · ejecución {RUN_ID} · Lucía {LUCIA}")

## 2. Lucía en memoria

Es el escenario L del notebook 07: un martes por la tarde llega Lucía y, a las 17:08, otro
paciente. Seis hechos verdaderos; la celda fuerza **a mano** lo que en una ejecución sorteará el
dado: el triaje se anota 7 min tarde, con el nivel equivocado (3 → 4), y alguien lo corrige
135 min después; la valoración sale dos veces (doble clic); el alta no se anota nunca.

| Registro | Qué es | Qué tendrá que hacer la limpieza |
| --- | --- | --- |
| 1, 2 | La llegada y el registro de Lucía | Nada |
| 3 | La llegada del otro paciente | Nada |
| 4 | El triaje: «hecho a las 17:06:12» (el PC de triaje va 2 min adelantado) y nivel 4 | Aplicar la corrección; el reloj, no se puede arreglar |
| 5, 6 | La valoración, dos veces (la segunda 30 s después) | Quitar la copia |
| 7 | `record_corrected`: el nivel del registro 4 era 3 | No es un hecho: solo sirve para corregir |
| — | El alta, que nadie anotó | Nada que hacer: Lucía será un **fantasma** |

La única diferencia con el 07 es el identificador: aquí la verdad de Lucía es `LUCIA`, propio de
esta ejecución del notebook, para que no choque con nada en Postgres.

In [ ]:
HORA = lambda instante: instante.astimezone(CET).strftime("%H:%M:%S")  # noqa: E731

# La verdad: el log del escenario L, con el catálogo de eventos de la versión 4.
sumidero_l = MemorySink()
log_l = EventLog(LUCIA, date(2026, 1, 6), sumidero_l, schema_version=4)
log_l.emit("patient_arrived", 1020.0, arrival_id=1, medio="propios_medios")
log_l.emit("patient_registered", 1022.5, arrival_id=1, edad=54, sexo="mujer")
log_l.emit("triage_completed", 1024.2, arrival_id=1, nivel=3, motivo="dolor_toracico")
log_l.emit("patient_arrived", 1028.0, arrival_id=2, medio="ambulancia")
log_l.emit("assessment_started", 1060.0, arrival_id=1, tipo="inicial", pool="medicos", box=1)
log_l.emit("patient_departed", 1135.0, arrival_id=1, destino="alta")
VERDAD_L = sumidero_l.events

CAMPOS = [campo.name for campo in dataclasses.fields(Roll)]


def tirada(**cambios):
    """Una tirada neutra (u = 0,99: ningún despiste; z = 0: la mediana) con los cambios dados."""
    valores = {c: 0.0 if c.startswith("z_") else 0.99 for c in CAMPOS}
    return Roll(**{**valores, **cambios})


class DadoAMano:
    """Un dado que da las tiradas dadas, en orden: lo único que `record` pide es `roll()`."""

    def __init__(self, tiradas):
        self.tiradas = list(tiradas)

    def roll(self):
        return self.tiradas.pop(0)


dado = DadoAMano(
    [
        tirada(),  # L1: nada
        tirada(),  # L2: nada
        # L3: retraso, valor erróneo (u_wrong 0,005 < tasa 0,01 del realista) y su corrección
        tirada(u_delay=0.0, u_wrong=0.005, u_wrong_field=0.0, u_wrong_value=0.9,
               u_correction=0.0),
        tirada(),  # L4: nada
        tirada(u_duplicate=0.0),  # L5: doble clic
        tirada(u_forget=0.0),  # L6: olvidado
    ]
)  # fmt: skip
realista = PERFILES_REGISTRO["realista"]
PERFIL_LUCIA = dataclasses.replace(
    realista,
    delay={**realista.delay, "triaje": Timed(0.60, (7.0, 15.0))},
    correction=Timed(0.5, (135.0, 720.0)),
)
L = record(VERDAD_L, PERFIL_LUCIA, 7, horizon=1440.0, run_id=LUCIA, dice=dado)
EPISODIO_LUCIA = VERDAD_L[0].episode_id

print(f"LO REGISTRADO  ({L.run_id}, en el orden en que se anota)")
for numero, registro in enumerate(L.records, start=1):
    print(f"  {numero} {registro.event_type:<20} {registro.episode_id}  "
          f"hecho {HORA(datetime.fromisoformat(registro.event_time))}  "
          f"anotado {HORA(datetime.fromisoformat(registro.recorded_time))}  "
          f"{json.loads(record_json(registro))['payload']}")  # fmt: skip
print("\nLA LIBRETA")
for fila in L.notebook:
    print(f"  {fila.kind:<14} {fila.wrong_field or ''} {fila.true_value or ''}")

# Las mismas horas que el notebook 07: el escenario L no ha cambiado.
assert [HORA(datetime.fromisoformat(r.recorded_time)) for r in L.records] == [
    "17:00:00", "17:02:30", "17:08:00", "17:13:12", "17:40:00", "17:40:30", "19:28:12",
]  # fmt: skip
assert Counter(fila.kind for fila in L.notebook) == {
    "bien": 4, "con_despistes": 1, "duplicado": 1, "correccion": 1, "perdido": 1,
}  # fmt: skip

## 3. La carga: bronce sin Kafka, la verdad y las cuatro construcciones

**Solo en modo normal.** La celda siguiente hace, en orden:

1. **Una ejecución de 14 días** en una carpeta temporal (`simulate_run`) y lo registrado con los
   tres perfiles (`record_run`): `limpio` sirve para comprobar que la limpieza no estropea
   nada (limpio − verdad = 0); `realista` es el caso normal; `hostil`, donde la limpieza tiene
   trabajo.
2. **Bronce, sin Kafka.** Lo registrado de los tres perfiles y lo de Lucía se escriben en bronce
   con `bronze_messages` y `write_bronze` (`edsim.store`): lo mismo que dejaría el consumidor
   tras una reemisión, con su `inicio`, sus registros y su `fin`, pero **con el calentamiento
   entero**. Una reemisión no basta: del calentamiento solo publica los episodios abiertos en W
   **según la verdad**, así que un paciente cuya alta se olvidó el primer día (un fantasma para
   el hospital) no llegaría a bronce, y el censo de lo registrado saldría más pequeño que en un
   hospital de verdad.
3. **La verdad en Postgres**, con la CLI: `edsim almacen cargar-verdad` (eventos, episodios,
   días y la libreta de cada perfil).
4. **Las cuatro construcciones de dbt**, como las lanzaría un analista (`AGENTS.md`): la hoja
   del hospital (lo registrado, limpio), la de la verdad y la ingenua (lo registrado sin
   limpiar), cada una en sus esquemas, y después la tabla de tres columnas. Sin los *unit tests*,
   que son pruebas y no corren contra la base `edsim`; con los tests de datos.

In [ ]:
def edsim(*argumentos):
    """Ejecuta la CLI del mismo entorno que el kernel; para el notebook si falla."""
    resultado = subprocess.run(
        [str(EDSIM), *map(str, argumentos)], cwd=RAIZ, capture_output=True, text=True, check=False
    )
    if resultado.returncode != 0:
        raise RuntimeError(
            f"edsim {' '.join(map(str, argumentos))}\n{resultado.stdout}{resultado.stderr}"
        )
    return resultado.stdout


def entorno_dbt():
    """El entorno de dbt: sin el VIRTUAL_ENV del núcleo, y sin las variables EDSIM_PG_* y DBT_*
    que pudieran venir de la terminal, que lo llevarían a otra base distinta de POSTGRES."""
    return {
        clave: valor
        for clave, valor in dbt_environment(os.environ).items()
        if not clave.startswith(("EDSIM_PG_", "DBT_"))
    }


def dbt(*argumentos):
    """`uv run dbt …` en analitica/, en su propio entorno, contra la base edsim."""
    resultado = subprocess.run(
        [UV, "run", "--locked", "--project", str(ANALITICA), "dbt", "--no-use-colors", *argumentos],
        cwd=ANALITICA,
        env=entorno_dbt(),
        capture_output=True,
        text=True,
        check=False,
    )
    if resultado.returncode != 0:
        raise RuntimeError(
            f"dbt {' '.join(argumentos)}\n{resultado.stdout[-6000:]}{resultado.stderr[-2000:]}"
        )
    return resultado.stdout


def consultar(sql, parametros=()):
    """Las filas de una consulta, con la conexión del paquete."""
    with connect(POSTGRES) as conexion:
        return conexion.execute(sql, parametros)


def publicar(info, lineas, *, calentamiento, horizonte):
    """Escribe en bronce una publicación entera, con la hora actual como primera marca."""
    mensajes = bronze_messages(
        info, lineas, warmup=calentamiento, horizon=horizonte, started_ms=time.time_ns() // 10**6
    )
    resultado = write_bronze(POSTGRES, mensajes)
    # Ni rechazados ni repetidos: cada mensaje entra una vez (los identificadores son nuevos).
    assert (resultado.saved, resultado.repeated, resultado.rejected) == (len(mensajes), 0, 0)
    return len(lineas)


TIEMPOS = {}
DERIVADO = {}
if RAPIDO:
    print("Modo rápido: sin Postgres ni dbt; esta celda se salta.")
else:
    # Las migraciones, antes de escribir nada (la segunda vez dice que ya estaban).
    print(edsim("almacen", "preparar", "--postgres", POSTGRES).strip().splitlines()[-1])
    CARPETA = tempfile.TemporaryDirectory(prefix="nb08-")
    ALMACEN = RunStore(Path(CARPETA.name))
    t = time.perf_counter()
    simulate_run(
        CONFIG / "centro.toml",
        CONFIG / "llegadas.toml",
        DIAS,
        SEMILLA,
        output_dir=Path(CARPETA.name) / RUN_ID,
        run_id=RUN_ID,
        warmup_days=CAL,
        origin="cli",
        profiles=CONFIG / "perfiles.toml",
    )
    TIEMPOS["simulación"] = time.perf_counter() - t

    t = time.perf_counter()
    for perfil in PERFILES:
        record_run(ALMACEN, RUN_ID, perfil, SEMILLA, profiles_file=CONFIG / "registro_clinico.toml")
    GRABADO = {perfil: read_recorded_run(ALMACEN, RUN_ID, perfil, SEMILLA) for perfil in PERFILES}
    TIEMPOS["registro (tres perfiles)"] = time.perf_counter() - t

    t = time.perf_counter()
    filas_bronce = 0
    for perfil, grabado in GRABADO.items():
        registros = read_recorded_records(grabado)
        DERIVADO[perfil] = grabado.derived_run_id
        info = RunInfo(
            run_id=grabado.derived_run_id,
            mode=REPLAY,
            schema_version=registros[0].schema_version,
            seed=SEMILLA,
            start_date=log_start_date(grabado.truth[0]),
            anchored=False,
            record=RecordInfo(RUN_ID, perfil, SEMILLA),
        )
        filas_bronce += publicar(
            info,
            [record_json(r) for r in registros],
            calentamiento=grabado.warmup,
            horizonte=grabado.horizon,
        )
    # Lucía: su día entero, W = 0 y H = 1440, con el bloque registro de su verdad.
    info_lucia = RunInfo(
        run_id=L.run_id,
        mode=REPLAY,
        schema_version=4,
        seed=7,
        start_date=date(2026, 1, 6),
        anchored=False,
        record=RecordInfo(LUCIA, "realista", 7),
    )
    filas_bronce += publicar(
        info_lucia, [record_json(r) for r in L.records], calentamiento=0.0, horizonte=1440.0
    )
    TIEMPOS["bronce (sin Kafka)"] = time.perf_counter() - t

    t = time.perf_counter()
    cargada = edsim(
        "almacen", "cargar-verdad", RUN_ID, "--ejecuciones", CARPETA.name, "--postgres", POSTGRES
    )
    print(cargada.strip().splitlines()[-1])
    TIEMPOS["cargar-verdad"] = time.perf_counter() - t

    SIN_UNIT = ("--exclude-resource-type", "unit_test")
    CONSTRUCCIONES = {
        "hoja hospital": ("build", "--exclude", "tag:laboratorio", *SIN_UNIT),
        "hoja verdad": ("build", "--exclude", "tag:laboratorio", "--vars", "{hoja: verdad}",
                        *SIN_UNIT),
        "hoja ingenuo": ("build", "--exclude", "tag:laboratorio", "--vars", "{hoja: ingenuo}",
                         *SIN_UNIT),
        "tres columnas": ("build", "--select", "tag:laboratorio", *SIN_UNIT),
    }  # fmt: skip
    for nombre, argumentos in CONSTRUCCIONES.items():
        t = time.perf_counter()
        salida = dbt(*argumentos)
        TIEMPOS[f"dbt: {nombre}"] = time.perf_counter() - t
        print(f"{nombre:<14} {salida.strip().splitlines()[-1]}")

    print(f"\nEn bronce: {filas_bronce} registros (tres perfiles y Lucía)")
    for paso, segundos in TIEMPOS.items():
        print(f"  {paso:<26}{segundos:>7.1f} s")

## 4. Lucía atraviesa las capas

**Solo en modo normal.** Las mismas siete filas, capa a capa. Todas las consultas filtran por
`run_id`: la base tiene más ejecuciones.

**Bronce** guarda lo que llegó, tal cual: los siete registros, también la copia del doble clic
y la corrección.

In [ ]:
LUCIA_REG = L.run_id
if RAPIDO:
    print("Modo rápido: sin Postgres; esta celda se salta.")
else:
    bronce_l = consultar(
        "SELECT event_type, episode_id, event_time, recorded_time, payload FROM bronce.eventos"
        " WHERE run_id = %s ORDER BY ingest_time",
        (LUCIA_REG,),
    )
    for tipo, episodio, hecho, anotado, payload in bronce_l:
        print(f"  {tipo:<20} {episodio}  hecho {HORA(hecho)}  anotado {HORA(anotado)}  {payload}")
    assert len(bronce_l) == len(L.records) == 7  # bronce no quita nada

**La preparación.** `preparacion.registros` deja una fila por registro (quita las relecturas de
Kafka, el mismo `event_id` dos veces; aquí no hay). `preparacion.registros_vigentes` aplica la
**limpieza conservadora**, tres reglas en este orden:

1. los `record_corrected` no son hechos: solo sirven para la regla 3;
2. **duplicados**: entre los registros con el mismo episodio, tipo, hora del hecho y `payload`,
   se queda el de menor `recorded_time` (el primero que se anotó);
3. **correcciones**: la última corrección de cada registro se aplica sobre su `payload`.

Lucía pasa de 7 filas a 6: sin la corrección y sin la copia de la valoración, y con el triaje
en nivel 3.

In [ ]:
if RAPIDO:
    print("Modo rápido: sin Postgres; esta celda se salta.")
else:
    vigentes_l = consultar(
        "SELECT event_type, episode_id, event_time, payload ->> 'nivel', corregido,"
        " duplicados_quitados FROM preparacion.registros_vigentes"
        " WHERE run_id = %s ORDER BY recorded_time",
        (LUCIA_REG,),
    )
    print(f"  {'tipo':<20} {'episodio':<12} {'hecho':>8} nivel corregido quitados")
    for tipo, episodio, hecho, nivel, corregido, quitados in vigentes_l:
        print(f"  {tipo:<20} {episodio:<12} {HORA(hecho):>8} {nivel or '':>5} "
              f"{corregido!s:>9} {quitados:>8}")  # fmt: skip
    tipos = [fila[0] for fila in vigentes_l]
    assert "record_corrected" not in tipos and tipos.count("assessment_started") == 1
    # El triaje, corregido: sin la regla 3 tendría nivel 4.
    ((nivel, corregido),) = [(f[3], f[4]) for f in vigentes_l if f[0] == "triage_completed"]
    assert (nivel, corregido) == ("3", True)
    # La valoración que queda dice que quitó una copia: la que se anotó 30 s después.
    assert [f[5] for f in vigentes_l if f[0] == "assessment_started"] == [1]

**Plata** tiene la forma de un HIS: una tabla por cosa, con su grano. Lucía tiene un triaje
(con nivel 3, y la hora que dio su ordenador: **17:06:12**, 2 min adelantada, porque el reloj no
se arregla), **una** valoración (sin su fin: el escenario no lo tiene) y un episodio con la señal
`sin_salida`: nadie anotó el alta.

In [ ]:
if RAPIDO:
    print("Modo rápido: sin Postgres; esta celda se salta.")
else:
    ((fin, nivel, corregido),) = consultar(
        "SELECT fin, nivel, corregido FROM plata.triaje WHERE run_id = %s AND episode_id = %s",
        (LUCIA_REG, EPISODIO_LUCIA),
    )
    valoraciones = consultar(
        "SELECT n, tipo, inicio, sin_pareja FROM plata.valoracion"
        " WHERE run_id = %s AND episode_id = %s",
        (LUCIA_REG, EPISODIO_LUCIA),
    )
    ((llegada, salida, sin_salida),) = consultar(
        "SELECT llegada, salida, sin_salida FROM plata.episodio"
        " WHERE run_id = %s AND episode_id = %s",
        (LUCIA_REG, EPISODIO_LUCIA),
    )
    print(f"triaje:     fin {HORA(fin)} · nivel {nivel} · corregido {corregido}")
    for n, tipo, inicio, sin_pareja in valoraciones:
        print(f"valoración: n {n} · {tipo} · inicio {HORA(inicio)} · sin_pareja {sin_pareja}")
    print(f"episodio:   llegada {HORA(llegada)} · salida {salida} · sin_salida {sin_salida}")
    assert nivel == 3  # pilla una corrección sin aplicar
    assert HORA(fin) == "17:06:12"  # el reloj del triaje sigue 2 min adelantado
    assert len(valoraciones) == 1  # pilla un doble clic sin quitar
    assert sin_salida and salida is None

**Oro** cuenta. El censo (`oro.censo`) son los pacientes dentro a cada hora en punto: quien llega
justo a la hora ya está. Lucía entra a las 17:00:00 justas, y el otro paciente a las 17:08, así
que cuenta desde las 18:00. Y como nadie anotó sus salidas, **los dos siguen dentro hasta el
final del día**: son fantasmas. En la verdad, Lucía salió a las 18:55.

In [ ]:
if RAPIDO:
    print("Modo rápido: sin Postgres; esta celda se salta.")
else:
    censo_l = consultar("SELECT k, censo FROM oro.censo WHERE run_id = %s ORDER BY k", (LUCIA_REG,))
    print("  hora  " + " ".join(f"{k:>2}" for k, _ in censo_l))
    print("  censo " + " ".join(f"{c:>2}" for _, c in censo_l))
    # 0 hasta las 16 h, 1 a las 17 h (Lucía), 2 de las 18 a las 23 h: el fantasma no se va.
    assert [c for _, c in censo_l] == [0] * 17 + [1] + [2] * 6

> **Lo que debes recordar:** cada capa hace una cosa. Bronce guarda lo que llega; la preparación
> quita relecturas y dobles clics y aplica correcciones; plata da la forma de un HIS, con señales
> como `sin_salida`; oro cuenta. Y hay cosas que ninguna capa puede saber: la hora real de un PC
> mal sincronizado o un alta que nadie anotó.

## 5. La tabla de tres columnas

**Solo en modo normal.** Para cada indicador del hito 7 (las llegadas, el censo medio, las
medianas hasta el triaje y hasta el médico y el reparto por nivel, en la ventana sin el
calentamiento), `laboratorio.tres_columnas` pone tres valores de la misma ejecución:

- **verdad**: con la verdad (la hoja `verdad`);
- **ingenuo**: con lo registrado, sin limpiar (la hoja `ingenuo`, como `compare` del hito 7);
- **limpio**: con lo registrado, limpiado (la hoja del hospital, la de Grafana).

`ingenuo − verdad` es cuánto se equivoca un indicador con datos sucios; `limpio − verdad`, lo
que queda tras la limpieza. Lo registrado se enlaza con su verdad por el bloque `registro` del
`inicio` de bronce. Con el perfil `limpio` las dos diferencias tienen que ser **0**.

In [ ]:
KPIS = (
    "llegadas", "censo_medio", "mediana_triaje_min", "mediana_medico_min",
    *(f"pct_nivel_{n}" for n in range(1, 6)),
)  # fmt: skip
TRES = {}
if RAPIDO:
    print("Modo rápido: sin Postgres; esta celda se salta.")
else:
    filas = consultar(
        "SELECT perfil, kpi, verdad, ingenuo, limpio, ingenuo_menos_verdad, limpio_menos_verdad"
        " FROM laboratorio.tres_columnas WHERE run_id = %s ORDER BY perfil, orden",
        (RUN_ID,),
    )
    for perfil, kpi, *valores in filas:
        TRES[perfil, kpi] = dict(
            zip(("verdad", "ingenuo", "limpio", "ingenuo-verdad", "limpio-verdad"), valores,
                strict=True)
        )  # fmt: skip
    for perfil in PERFILES:
        print(f"\n{perfil}")
        print(f"  {'kpi':<20}{'verdad':>10}{'ingenuo':>10}{'limpio':>10}"
              f"{'ing-ver':>10}{'lim-ver':>10}")  # fmt: skip
        for kpi in KPIS:
            v = TRES[perfil, kpi]
            print(f"  {kpi:<20}" + "".join(f"{x:>10.2f}" for x in v.values()))
    assert sorted({perfil for perfil, _ in TRES}) == sorted(PERFILES)
    # Con el perfil limpio, las dos diferencias son 0 en los nueve: pilla mezclar ejecuciones o
    # una regla de la limpieza que estropea datos limpios.
    for kpi in KPIS:
        assert TRES["limpio", kpi]["ingenuo-verdad"] == TRES["limpio", kpi]["limpio-verdad"] == 0
    # Un doble clic no cambia la primera llegada ni la primera valoración de un episodio: la
    # limpieza no mueve el censo ni las medianas (lo que los separa de la verdad, en la sección 6).
    for perfil in PERFILES[1:]:
        for kpi in ("censo_medio", "mediana_triaje_min", "mediana_medico_min"):
            assert TRES[perfil, kpi]["ingenuo"] == TRES[perfil, kpi]["limpio"], (perfil, kpi)

La gráfica compara, en los perfiles `realista` y `hostil`, cuánto se separa cada indicador de
la verdad antes y después de limpiar. Las unidades no son las mismas (llegadas, pacientes,
minutos y puntos porcentuales): mira el tamaño de cada par de barras, no el de unas frente a
otras.

In [ ]:
if RAPIDO:
    print("Modo rápido: sin Postgres; esta celda se salta.")
else:
    fig, ejes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
    y = range(len(KPIS))
    for eje, perfil in zip(ejes, PERFILES[1:], strict=True):
        ingenuo = [TRES[perfil, kpi]["ingenuo-verdad"] for kpi in KPIS]
        limpio = [TRES[perfil, kpi]["limpio-verdad"] for kpi in KPIS]
        eje.barh([k + 0.2 for k in y], ingenuo, height=0.4, label="ingenuo − verdad",
                 color=MUTED_COLOR)  # fmt: skip
        eje.barh([k - 0.2 for k in y], limpio, height=0.4, label="limpio − verdad",
                 color=COLOR[perfil])  # fmt: skip
        eje.axvline(0, color="black", linewidth=0.8)
        eje.set_xscale("symlog", linthresh=1)
        eje.set_title(perfil)
        eje.spines[["top", "right"]].set_visible(False)
    ejes[0].set_yticks(list(y), KPIS)
    ejes[0].invert_yaxis()
    ejes[1].legend(frameon=False)
    fig.suptitle("Cuánto se separa cada indicador de la verdad, antes y después de limpiar")
    fig.tight_layout()
    plt.show()

Lo que hace la limpieza depende del indicador:

- **Las llegadas cambian de signo.** Con el realista, lo ingenuo da +3, casi la verdad, pero
  **por casualidad**: cuenta dos veces los dobles clics de la admisión y no cuenta las llegadas
  que nadie anotó, y los dos errores se compensan. Lo limpio quita los dobles clics y deja a la
  vista el olvido: −15, exactamente las llegadas olvidadas en la ventana (sección 6). Un
  indicador que acierta por dos errores que se anulan es peor que uno que se equivoca por uno que
  se conoce.
- **El reparto por nivel se acerca a la verdad**, sobre todo con el hostil: la limpieza aplica
  las correcciones de nivel.
- **El censo y las medianas de tiempo no se mueven nada**: un doble clic no cambia la primera
  llegada ni la primera valoración de un episodio. Lo que los separa de la verdad (los fantasmas
  y los relojes) no lo arregla ninguna regla; la sección siguiente lo explica con la libreta.

> **Lo que debes recordar:** sin la verdad no se puede medir cuánto recupera la limpieza; con
> ella, cada diferencia que queda tiene una causa que la libreta dice.

## 6. Lo que no se arregla

**Solo en modo normal.** La limpieza es **conservadora**: solo arregla lo que se puede saber
desde los datos (una copia idéntica, una corrección que dice a qué registro corrige). Hay
despistes que dejan el dato con buena cara y que nadie puede deshacer:

- **El reloj desajustado.** Cada sistema escribe la hora con su reloj. Una mediana de tiempo
  resta dos horas de dos sistemas: hasta el triaje, la del PC de triaje menos la de admisión;
  hasta el médico, la del HIS menos la de admisión. Así que `limpio − verdad` debe ser, casi
  exactamente, **la resta de los dos relojes**, como en el notebook 07.
- **Un olvido.** Una llegada que nadie anotó no se cuenta. Y una salida que nadie anotó deja un
  **fantasma** que infla el censo hasta el final.
- **Un valor erróneo que nadie corrigió.** Un nivel mal tecleado sin corrección sigue mal.

La libreta de cada perfil (`defectos.csv`) dice qué le pasó a cada hecho: con ella se explica
cada diferencia.

In [ ]:
if RAPIDO:
    print("Modo rápido: sin Postgres; esta celda se salta.")
else:
    for perfil in PERFILES[1:]:
        reloj = PERFILES_REGISTRO[perfil].clock
        print(f"{perfil}: relojes {dict(reloj)}")
        for kpi, sistema in (("mediana_triaje_min", "triaje"), ("mediana_medico_min", "his")):
            diferencia = TRES[perfil, kpi]["limpio-verdad"]
            esperado = reloj[sistema] - reloj["admision"]
            print(f"  {kpi:<20} limpio − verdad {diferencia:+7.3f} min · reloj {sistema} "
                  f"menos admisión {esperado:+g} min")  # fmt: skip
            # La tolerancia de 1 min es la del 07: los hechos olvidados o vacíos mueven algo
            # la mediana. Sin el reloj, la diferencia sería casi 0.
            assert abs(diferencia - esperado) < 1.0, (perfil, kpi)

**Las llegadas olvidadas.** Lo limpio cuenta un paciente por llegada anotada (sin las copias),
así que le faltan las llegadas que nadie anotó **dentro de la ventana**. Con el perfil
`realista`, el reloj de la admisión va en hora, y la diferencia es exactamente esa. Con el
`hostil`, ese reloj va 3 min adelantado: alguna llegada de los últimos minutos del calentamiento
«salta» dentro de la ventana, y la cuenta se separa en una o dos.

La misma celda cuenta las **salidas olvidadas**: cada una deja un fantasma que el censo cuenta
hasta el final de la ventana. Con el realista, unas 25 bastan para subir el censo medio en unos
11 pacientes (de 39); con el hostil, el censo se multiplica por cuatro.

In [ ]:
SALIDAS = ("patient_departed", "patient_left_wbs")
if RAPIDO:
    print("Modo rápido: sin Postgres; esta celda se salta.")
else:
    for perfil in PERFILES[1:]:
        grabado = GRABADO[perfil]
        hecho = {evento.event_id: evento for evento in grabado.truth}
        perdidos = [hecho[n.fact_id] for n in grabado.notebook if n.kind == "perdido"]
        llegadas = [e for e in perdidos if e.event_type == "patient_arrived"]
        en_ventana = [e for e in llegadas if e.sim_time >= CALENTAMIENTO]
        salidas = [e for e in perdidos if e.event_type in SALIDAS]
        diferencia = TRES[perfil, "llegadas"]["limpio-verdad"]
        print(f"{perfil}: llegadas limpio − verdad {diferencia:+.0f} · olvidadas en la ventana "
              f"{len(en_ventana)} (en toda la ejecución, {len(llegadas)}) · salidas olvidadas "
              f"{len(salidas)} · censo limpio − verdad "
              f"{TRES[perfil, 'censo_medio']['limpio-verdad']:+.2f}")  # fmt: skip
        if PERFILES_REGISTRO[perfil].clock["admision"] == 0:
            # Exacto con el reloj de admisión en hora: pilla una llegada perdida por la limpieza.
            assert diferencia == -len(en_ventana)

**Los valores erróneos sin corregir.** La libreta apunta cada valor erróneo y cada corrección:
los que no tienen corrección siguen mal en lo limpio. En el reparto por nivel pesan los niveles
mal tecleados; el resto (edad, motivo y destino) no entra en los cinco indicadores, pero un
analista los vería en plata.

In [ ]:
if RAPIDO:
    print("Modo rápido: sin Postgres; esta celda se salta.")
else:
    for perfil in PERFILES[1:]:
        notas = GRABADO[perfil].notebook
        corregidos = {n.corrects for n in notas if n.kind == "correccion"}
        erroneos = [n for n in notas if n.kind == "con_despistes" and n.wrong_field]
        sin_corregir = Counter(n.wrong_field for n in erroneos if n.record_id not in corregidos)
        por_campo = dict(sorted(sin_corregir.items()))
        print(f"{perfil}: {len(erroneos)} valores erróneos, {len(corregidos)} corregidos; "
              f"sin corregir {sum(sin_corregir.values())}: {por_campo}")  # fmt: skip
        assert sum(sin_corregir.values()) == len(erroneos) - len(corregidos)

> **Lo que debes recordar:** lo que queda en `limpio − verdad` no es ruido: es el reloj (en las
> medianas), los olvidos (en las llegadas y, con los fantasmas, en el censo) y los valores
> erróneos sin corrección (en el reparto por nivel). La limpieza no los puede ver, porque el
> dato tiene buena cara.

## 7. Cada regla frente a la libreta

**Solo en modo normal.** `oro.calidad` cuenta, por ejecución, lo que hizo la limpieza y lo que
le parece raro. La libreta dice lo que hizo de verdad el registro clínico. Cada regla tiene su
relación:

| Regla | En `oro.calidad` | En la libreta | Relación |
| --- | --- | --- | --- |
| Duplicados | `duplicados_quitados` | filas `duplicado` | Iguales, y además son **los mismos registros** |
| Correcciones | `registros_corregidos` | filas `correccion` | Iguales |
| Correcciones sin su registro | `correcciones_sin_registro` | — | 0 |
| Campos vacíos (no se arreglan) | `campos_vacios` | filas con `campo_vacio` | Iguales |

¿Por qué no hay correcciones sin registro? Una corrección siempre apunta al registro erróneo
original; su copia de un doble clic se anota después, y la regla 2 se queda con el primero que
se anotó. Y un hecho olvidado no tiene registro erróneo que corregir. En un hospital real sí
podría pasar (alguien corrige la copia), y por eso `oro.calidad` lo cuenta.

In [ ]:
COPIAS_QUITADAS = """
SELECT r.event_id FROM preparacion.registros AS r
WHERE r.run_id = %s AND r.event_type <> 'record_corrected' AND r.episode_id IS NOT NULL
  AND NOT EXISTS (SELECT 1 FROM preparacion.registros_vigentes AS v
                  WHERE v.run_id = r.run_id AND v.event_id = r.event_id)
"""
if RAPIDO:
    print("Modo rápido: sin Postgres; esta celda se salta.")
else:
    columnas = ("duplicados_quitados", "registros_corregidos", "correcciones_sin_registro",
                "campos_vacios")  # fmt: skip
    print(f"{'':<10}" + "".join(f"{c:>28}" for c in columnas))
    for perfil in PERFILES:
        notas = GRABADO[perfil].notebook
        clases = Counter(n.kind for n in notas)
        vacios = sum(1 for n in notas if n.kind == "con_despistes" and n.empty_field)
        ((quitados, corregidos, sin_registro, campos_vacios),) = consultar(
            f"SELECT {', '.join(columnas)} FROM oro.calidad WHERE run_id = %s",
            (DERIVADO[perfil],),
        )
        libreta = (clases["duplicado"], clases["correccion"], 0, vacios)
        oro = (quitados, corregidos, sin_registro, campos_vacios)
        celdas = [f"{de_oro} (libreta {de_libreta})" for de_oro, de_libreta in
                  zip(oro, libreta, strict=True)]  # fmt: skip
        print(f"{perfil:<10}" + "".join(f"{celda:>28}" for celda in celdas))
        # Criterio de aceptación del hito: los duplicados quitados son los de la libreta. Pilla
        # una limpieza que quita de más (un hecho real) o de menos.
        assert oro == libreta, perfil
        # Y no solo cuántos: los registros quitados son las copias que apuntó la libreta.
        quitadas = {event_id for (event_id,) in consultar(COPIAS_QUITADAS, (DERIVADO[perfil],))}
        assert quitadas == {n.record_id for n in notas if n.kind == "duplicado"}, perfil

> **Lo que debes recordar:** la libreta es el solucionario. Con ella cada regla de la limpieza se
> comprueba registro a registro: quita exactamente los dobles clics, aplica todas las
> correcciones, y deja a la vista (sin inventarse nada) lo que no puede arreglar.

## 8. Resumen

1. dbt transforma bronce en capas: la preparación limpia (relecturas, dobles clics y
   correcciones), plata da la forma de un HIS y oro cuenta.
2. La limpieza es conservadora: arregla lo que se puede saber desde los datos y deja señales
   (`sin_salida`, `oro.calidad`) de lo demás.
3. La tabla de tres columnas, posible solo con la verdad, mide cuánto se equivoca cada indicador
   con datos sucios y cuánto recupera la limpieza.
4. Lo que queda es el reloj, los olvidos y los valores erróneos sin corregir: cada diferencia
   tiene su causa en la libreta.
5. Cada regla cuadra con la libreta registro a registro.

La celda siguiente imprime la consulta para borrar de `edsim` lo que escribió este notebook
(no la ejecuta): después, una construcción nueva de dbt deja plata y oro sin sus filas.

In [ ]:
if not RAPIDO:
    registrados = [*DERIVADO.values(), LUCIA_REG]
    ((filas,),) = consultar(
        "SELECT count(*) FROM bronce.eventos WHERE run_id = ANY(%s)", (registrados,)
    )
    print(f"El notebook dejó en bronce.eventos {filas} filas. Para borrarlas, en psql:\n")
    lista = ", ".join(f"'{r}'" for r in registrados)
    print(f"DELETE FROM bronce.eventos WHERE run_id IN ({lista});")
    print(f"DELETE FROM bronce.ejecuciones WHERE run_id IN ({lista});")
    for tabla in ("eventos", "episodios", "dias"):
        print(f"DELETE FROM verdad.{tabla} WHERE run_id = '{RUN_ID}';")
    # La libreta va por el run_id de lo registrado; el de la verdad está en run_id_origen.
    print(f"DELETE FROM verdad.libreta WHERE run_id_origen = '{RUN_ID}';")
    CARPETA.cleanup()  # la carpeta temporal de la sección 3
print(f"Tiempo total: {time.perf_counter() - INICIO:.0f} s")